# Smart Campus Tour & Information Multi-Modal Chatbot - Google Colab Version

        This notebook recreates the Streamlit project inside Google Colab. Run the cells from top to bottom.

        It creates the synthetic campus dataset, SQLite knowledge base, FAQ dataset, Colab Streamlit app, evaluation scenarios, and a public tunnel URL.

## 1. Install Dependencies

        The full install includes Whisper, PyTorch, Transformers, SentenceTransformers, FAISS, OpenCV, Librosa, and Streamlit.
        This can take several minutes on first run.

In [ ]:
        !apt-get -qq update
        !apt-get -qq install -y ffmpeg
        !pip -q install streamlit pandas numpy pillow matplotlib scikit-learn
        !pip -q install torch transformers sentence-transformers faiss-cpu openai-whisper librosa soundfile opencv-python-headless wordcloud
        

## 2. Create Synthetic Campus Dataset and SQLite Knowledge Base

In [ ]:
        import json, sqlite3, random
        from pathlib import Path
        import pandas as pd
        import numpy as np

        BASE = Path('/content/smart_campus_colab')
        DATA_DIR = BASE / 'data'
        REPORTS_DIR = BASE / 'reports'
        FIGURES_DIR = REPORTS_DIR / 'figures'
        RESULTS_DIR = REPORTS_DIR / 'results'
        for path in [DATA_DIR / 'campus_images', DATA_DIR / 'audio_queries', FIGURES_DIR, RESULTS_DIR]:
            path.mkdir(parents=True, exist_ok=True)

        locations = [
  {
    "id": 1,
    "name": "Library",
    "category": "Learning",
    "description": "A four-floor academic library with silent study rooms, group booths, assistive technology, archives, and a staffed research help desk.",
    "opening_hours": "Mon-Fri 08:00-22:00; Sat-Sun 10:00-18:00",
    "coordinates": {
      "lat": 51.7538,
      "lon": -1.2541
    },
    "events": [
      "Referencing workshop every Tuesday",
      "Exam revision clinic in May",
      "Digital archives showcase"
    ]
  },
  {
    "id": 2,
    "name": "Student Union",
    "category": "Student Services",
    "description": "Central student life building with societies office, advice centre, event hall, student shop, and wellbeing drop-in spaces.",
    "opening_hours": "Mon-Fri 09:00-20:00; Sat 11:00-16:00; Sun closed",
    "coordinates": {
      "lat": 51.7545,
      "lon": -1.2552
    },
    "events": [
      "Freshers fair",
      "Society showcase",
      "International student mixer"
    ]
  },
  {
    "id": 3,
    "name": "Cafeteria",
    "category": "Food",
    "description": "Main campus dining venue serving breakfast, hot lunches, vegetarian meals, halal options, coffee, and grab-and-go snacks.",
    "opening_hours": "Mon-Fri 07:30-19:00; Sat-Sun 09:00-15:00",
    "coordinates": {
      "lat": 51.7541,
      "lon": -1.2531
    },
    "events": [
      "World food week",
      "Student meal deal launch",
      "Sustainability lunch talk"
    ]
  },
  {
    "id": 4,
    "name": "Computer Lab",
    "category": "Learning",
    "description": "Specialist computing lab with GPU workstations, Python environments, VR kits, printing, and technical support for AI modules.",
    "opening_hours": "Mon-Fri 08:00-21:00; Sat 10:00-17:00; Sun closed",
    "coordinates": {
      "lat": 51.755,
      "lon": -1.2526
    },
    "events": [
      "Python bootcamp",
      "Data science clinic",
      "Hackathon preparation session"
    ]
  },
  {
    "id": 5,
    "name": "Lecture Hall A",
    "category": "Teaching",
    "description": "Large tiered lecture theatre with 300 seats, induction loop, dual projection, lecture capture, and accessible front-row seating.",
    "opening_hours": "Mon-Fri 08:00-18:00; weekend by booking",
    "coordinates": {
      "lat": 51.7549,
      "lon": -1.2546
    },
    "events": [
      "Guest AI ethics lecture",
      "Undergraduate open day talk",
      "Public policy debate"
    ]
  },
  {
    "id": 6,
    "name": "Lecture Hall B",
    "category": "Teaching",
    "description": "Medium lecture theatre for seminars, flipped classroom sessions, guest talks, and evening public engagement events.",
    "opening_hours": "Mon-Fri 08:30-19:00; weekend by booking",
    "coordinates": {
      "lat": 51.7553,
      "lon": -1.2544
    },
    "events": [
      "Research methods seminar",
      "Careers panel",
      "Postgraduate welcome briefing"
    ]
  },
  {
    "id": 7,
    "name": "Sports Centre",
    "category": "Wellbeing",
    "description": "Fitness and wellbeing facility with gym, courts, climbing wall, sports therapy, changing rooms, and inclusive sport sessions.",
    "opening_hours": "Mon-Fri 06:30-22:00; Sat-Sun 08:00-20:00",
    "coordinates": {
      "lat": 51.7529,
      "lon": -1.2564
    },
    "events": [
      "Intramural finals",
      "Yoga for beginners",
      "Inclusive sports afternoon"
    ]
  },
  {
    "id": 8,
    "name": "Administration Office",
    "category": "Administration",
    "description": "Main student administration office for enrolment, transcripts, fees, ID cards, letters, and registry appointments.",
    "opening_hours": "Mon-Fri 09:00-17:00; Sat-Sun closed",
    "coordinates": {
      "lat": 51.7543,
      "lon": -1.2522
    },
    "events": [
      "Visa document drop-in",
      "Graduation briefing",
      "Registration support week"
    ]
  },
  {
    "id": 9,
    "name": "Innovation Hub",
    "category": "Enterprise",
    "description": "Collaborative maker and start-up space with prototyping equipment, meeting pods, incubator support, and entrepreneurship mentoring.",
    "opening_hours": "Mon-Fri 08:30-20:00; Sat 10:00-16:00; Sun closed",
    "coordinates": {
      "lat": 51.7558,
      "lon": -1.2535
    },
    "events": [
      "Start-up pitch night",
      "Design thinking sprint",
      "Prototype demo day"
    ]
  },
  {
    "id": 10,
    "name": "Career Centre",
    "category": "Student Services",
    "description": "Careers and employability centre offering CV checks, mock interviews, placement advice, employer fairs, and graduate job support.",
    "opening_hours": "Mon-Fri 09:00-18:00; Sat-Sun closed",
    "coordinates": {
      "lat": 51.7536,
      "lon": -1.2529
    },
    "events": [
      "Graduate careers fair",
      "LinkedIn clinic",
      "Mock assessment centre"
    ]
  },
  {
    "id": 11,
    "name": "Health Centre",
    "category": "Wellbeing",
    "description": "Campus health service with nurse appointments, mental health referrals, vaccination clinics, and accessible consultation rooms.",
    "opening_hours": "Mon-Fri 08:30-17:30; Sat-Sun closed",
    "coordinates": {
      "lat": 51.7526,
      "lon": -1.2538
    },
    "events": [
      "Flu vaccination clinic",
      "Mental health awareness week",
      "Sleep hygiene workshop"
    ]
  },
  {
    "id": 12,
    "name": "Engineering Building",
    "category": "Teaching",
    "description": "Engineering teaching and research building with electronics benches, robotics labs, design studios, and project demonstration spaces.",
    "opening_hours": "Mon-Fri 08:00-20:00; Sat 10:00-16:00; Sun closed",
    "coordinates": {
      "lat": 51.7562,
      "lon": -1.2519
    },
    "events": [
      "Robotics showcase",
      "Sustainable design review",
      "Industry guest lecture"
    ]
  },
  {
    "id": 13,
    "name": "Science Centre",
    "category": "Research",
    "description": "Science facility containing wet labs, microscopy suites, chemistry teaching rooms, safety training spaces, and research offices.",
    "opening_hours": "Mon-Fri 08:00-19:00; weekend restricted access",
    "coordinates": {
      "lat": 51.7566,
      "lon": -1.2528
    },
    "events": [
      "Lab safety induction",
      "Women in STEM seminar",
      "Research poster evening"
    ]
  },
  {
    "id": 14,
    "name": "Arts Studio",
    "category": "Creative",
    "description": "Creative arts studio with photography bays, editing suites, printmaking equipment, critique rooms, and exhibition wall space.",
    "opening_hours": "Mon-Fri 09:00-21:00; Sat 10:00-17:00; Sun closed",
    "coordinates": {
      "lat": 51.7531,
      "lon": -1.2517
    },
    "events": [
      "Student exhibition",
      "Portfolio review",
      "Photography masterclass"
    ]
  },
  {
    "id": 15,
    "name": "Music Hall",
    "category": "Creative",
    "description": "Performance venue with rehearsal rooms, recording booth, grand piano, sound desk, and evening concert programme.",
    "opening_hours": "Mon-Fri 09:00-22:00; Sat-Sun 10:00-18:00",
    "coordinates": {
      "lat": 51.7527,
      "lon": -1.2524
    },
    "events": [
      "Jazz evening",
      "Choir rehearsal",
      "Student composition showcase"
    ]
  },
  {
    "id": 16,
    "name": "Accommodation Office",
    "category": "Student Services",
    "description": "Housing support office for halls applications, maintenance reporting, private renting advice, and accommodation contracts.",
    "opening_hours": "Mon-Fri 09:00-17:00; Sat 10:00-14:00; Sun closed",
    "coordinates": {
      "lat": 51.7519,
      "lon": -1.2549
    },
    "events": [
      "Housing fair",
      "Private renting advice session",
      "Residence life welcome"
    ]
  },
  {
    "id": 17,
    "name": "Bookshop",
    "category": "Retail",
    "description": "Campus bookshop selling textbooks, stationery, lab notebooks, university merchandise, and course reading bundles.",
    "opening_hours": "Mon-Fri 09:00-18:00; Sat 10:00-16:00; Sun closed",
    "coordinates": {
      "lat": 51.754,
      "lon": -1.2557
    },
    "events": [
      "Author signing",
      "Second-hand textbook sale",
      "Reading list support desk"
    ]
  },
  {
    "id": 18,
    "name": "Prayer Room",
    "category": "Faith",
    "description": "Quiet multi-faith prayer and reflection room with washing facilities, privacy screens, shoe storage, and nearby pastoral support.",
    "opening_hours": "Daily 07:00-22:00",
    "coordinates": {
      "lat": 51.7533,
      "lon": -1.2535
    },
    "events": [
      "Interfaith dialogue",
      "Mindfulness session",
      "Chaplaincy coffee morning"
    ]
  },
  {
    "id": 19,
    "name": "Research Centre",
    "category": "Research",
    "description": "Postgraduate research centre with doctoral offices, project rooms, seminar spaces, ethics support, and interdisciplinary collaboration areas.",
    "opening_hours": "Mon-Fri 08:00-20:00; Sat 10:00-16:00; Sun closed",
    "coordinates": {
      "lat": 51.756,
      "lon": -1.2556
    },
    "events": [
      "PhD writing retreat",
      "Research ethics clinic",
      "Interdisciplinary symposium"
    ]
  },
  {
    "id": 20,
    "name": "Accessibility Office",
    "category": "Student Services",
    "description": "Accessibility and disability support office providing learning plans, assistive technology advice, campus access guidance, and exam adjustments.",
    "opening_hours": "Mon-Fri 09:00-17:00; Sat-Sun closed",
    "coordinates": {
      "lat": 51.7539,
      "lon": -1.2514
    },
    "events": [
      "Assistive technology demo",
      "Inclusive teaching forum",
      "Neurodiversity support group"
    ]
  }
]
        (DATA_DIR / 'campus_locations.json').write_text(json.dumps(locations, indent=2), encoding='utf-8')
        (DATA_DIR / 'campus_knowledge_export.json').write_text(json.dumps(locations, indent=2), encoding='utf-8')

        conn = sqlite3.connect(DATA_DIR / 'knowledge_base.db')
        conn.execute('''
        CREATE TABLE IF NOT EXISTS campus_locations (
            id INTEGER PRIMARY KEY,
            name TEXT,
            category TEXT,
            description TEXT,
            opening_hours TEXT,
            latitude REAL,
            longitude REAL,
            events TEXT
        )
        ''')
        for row in locations:
            conn.execute(
                'INSERT OR REPLACE INTO campus_locations VALUES (?, ?, ?, ?, ?, ?, ?, ?)',
                (
                    row['id'], row['name'], row['category'], row['description'],
                    row['opening_hours'], row['coordinates']['lat'], row['coordinates']['lon'],
                    json.dumps(row['events'])
                )
            )
        conn.commit()
        conn.close()

        intents = ['find_location', 'opening_hours', 'event_query', 'facility_information', 'study_space', 'food_services', 'accessibility', 'greeting', 'goodbye', 'other']
        rows = []
        for loc in locations:
            name = loc['name']
            rows += [
                {'text': f'Where is the {name}?', 'intent': 'find_location', 'location': name},
                {'text': f'How do I get to {name}?', 'intent': 'find_location', 'location': name},
                {'text': f'What time does {name} open?', 'intent': 'opening_hours', 'location': name},
                {'text': f'When does {name} close?', 'intent': 'opening_hours', 'location': name},
                {'text': f'What events are happening at {name}?', 'intent': 'event_query', 'location': name},
                {'text': f'Describe the facilities at {name}', 'intent': 'facility_information', 'location': name},
            ]
        extras = [
            ('Where can I find quiet study space?', 'study_space'),
            ('Where can I get vegetarian lunch?', 'food_services'),
            ('I need accessibility support', 'accessibility'),
            ('hello campus assistant', 'greeting'),
            ('goodbye and thanks', 'goodbye'),
            ('What is the weather today?', 'other'),
        ]
        while len(rows) < 320:
            text, intent = extras[len(rows) % len(extras)]
            rows.append({'text': text, 'intent': intent, 'location': ''})
        faq = pd.DataFrame(rows).sample(frac=1, random_state=42)
        faq.to_csv(DATA_DIR / 'faq_dataset.csv', index=False)
        print('Created Colab dataset at', BASE)
        print('FAQ rows:', len(faq))
        print('Locations:', len(locations))
        

## 3. Write the Streamlit App

In [ ]:
        from pathlib import Path
        BASE = Path('/content/smart_campus_colab')
        app_code = "\nfrom __future__ import annotations\n\nimport json\nimport sqlite3\nimport tempfile\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport streamlit as st\nfrom PIL import Image\n\nBASE = Path(\"/content/smart_campus_colab\")\nDATA_DIR = BASE / \"data\"\nDB_PATH = DATA_DIR / \"knowledge_base.db\"\nLOCATION_JSON = DATA_DIR / \"campus_locations.json\"\n\nst.set_page_config(page_title=\"Smart Campus Multimodal Assistant\", layout=\"wide\")\n\nst.markdown(\n    \"\"\"\n    <style>\n    html, body, .stApp, [data-testid=\"stAppViewContainer\"] {\n        background: #f7f9fb !important;\n        color: #102033 !important;\n    }\n    [data-testid=\"stHeader\"] { background: #10141a !important; }\n    section[data-testid=\"stSidebar\"] {\n        background: #102033 !important;\n        color: white !important;\n    }\n    section[data-testid=\"stSidebar\"] * { color: white !important; }\n    [data-testid=\"stMain\"] *, .block-container *, h1, h2, h3, p, span, label {\n        color: #102033 !important;\n    }\n    textarea, input {\n        background: #ffffff !important;\n        color: #102033 !important;\n        border: 1px solid #c7d3df !important;\n    }\n    .result-panel {\n        background: #ffffff;\n        border: 1px solid #d9e1ea;\n        border-radius: 8px;\n        padding: 1rem;\n        color: #102033 !important;\n    }\n    .result-panel * { color: #102033 !important; }\n    </style>\n    \"\"\",\n    unsafe_allow_html=True,\n)\n\n\ndef load_locations() -> list[dict]:\n    return json.loads(LOCATION_JSON.read_text(encoding=\"utf-8\"))\n\n\ndef connect() -> sqlite3.Connection:\n    conn = sqlite3.connect(DB_PATH)\n    conn.row_factory = sqlite3.Row\n    return conn\n\n\ndef init_db() -> None:\n    DATA_DIR.mkdir(parents=True, exist_ok=True)\n    with connect() as conn:\n        conn.execute(\n            \"\"\"\n            CREATE TABLE IF NOT EXISTS campus_locations (\n                id INTEGER PRIMARY KEY,\n                name TEXT,\n                category TEXT,\n                description TEXT,\n                opening_hours TEXT,\n                latitude REAL,\n                longitude REAL,\n                events TEXT\n            )\n            \"\"\"\n        )\n        for row in load_locations():\n            conn.execute(\n                \"\"\"\n                INSERT OR REPLACE INTO campus_locations\n                VALUES (?, ?, ?, ?, ?, ?, ?, ?)\n                \"\"\",\n                (\n                    row[\"id\"],\n                    row[\"name\"],\n                    row[\"category\"],\n                    row[\"description\"],\n                    row[\"opening_hours\"],\n                    row[\"coordinates\"][\"lat\"],\n                    row[\"coordinates\"][\"lon\"],\n                    json.dumps(row[\"events\"]),\n                ),\n            )\n        conn.commit()\n\n\ndef all_locations() -> list[dict]:\n    with connect() as conn:\n        rows = conn.execute(\"SELECT * FROM campus_locations ORDER BY id\").fetchall()\n    return [\n        {\n            \"id\": row[\"id\"],\n            \"name\": row[\"name\"],\n            \"category\": row[\"category\"],\n            \"description\": row[\"description\"],\n            \"opening_hours\": row[\"opening_hours\"],\n            \"coordinates\": {\"lat\": row[\"latitude\"], \"lon\": row[\"longitude\"]},\n            \"events\": json.loads(row[\"events\"]),\n        }\n        for row in rows\n    ]\n\n\ndef expand_query(query: str) -> str:\n    replacements = {\n        \"heppening\": \"happening\",\n        \"hapenning\": \"happening\",\n        \"eventhub\": \"innovation hub\",\n        \"event hub\": \"innovation hub\",\n        \"gym\": \"sports centre\",\n        \"it lab\": \"computer lab\",\n    }\n    out = query.lower()\n    for source, target in replacements.items():\n        out = out.replace(source, target)\n    return out\n\n\ndef intent(text: str) -> tuple[str, float]:\n    q = expand_query(text)\n    rules = [\n        (\"greeting\", [\"hello\", \"hi\", \"hey\"]),\n        (\"goodbye\", [\"bye\", \"thanks\", \"goodbye\"]),\n        (\"opening_hours\", [\"open\", \"close\", \"hours\", \"time\", \"weekend\"]),\n        (\"event_query\", [\"event\", \"events\", \"happening\", \"workshop\", \"fair\"]),\n        (\"food_services\", [\"food\", \"lunch\", \"coffee\", \"halal\", \"vegetarian\", \"cafeteria\"]),\n        (\"study_space\", [\"study\", \"quiet\", \"silent\", \"booth\", \"revise\"]),\n        (\"accessibility\", [\"accessibility\", \"disabled\", \"disability\", \"step free\", \"assistive\"]),\n        (\"find_location\", [\"where\", \"directions\", \"find\", \"get to\", \"take me\"]),\n        (\"facility_information\", [\"facility\", \"facilities\", \"describe\", \"information\"]),\n    ]\n    for label, keywords in rules:\n        if any(keyword in q for keyword in keywords):\n            return label, 0.74\n    return \"other\", 0.55\n\n\n@st.cache_resource(show_spinner=False)\ndef sentence_model():\n    try:\n        from sentence_transformers import SentenceTransformer\n\n        return SentenceTransformer(\"sentence-transformers/all-MiniLM-L6-v2\")\n    except Exception:\n        return None\n\n\ndef semantic_search(query: str, top_k: int = 3) -> list[dict]:\n    records = all_locations()\n    q = expand_query(query)\n    exact = []\n    for record in records:\n        if record[\"name\"].lower() in q:\n            exact.append({\"location\": record, \"score\": 1.0})\n    if exact:\n        remainder = [r for r in records if r[\"name\"] != exact[0][\"location\"][\"name\"]]\n        return exact + [{\"location\": r, \"score\": 0.2} for r in remainder[: top_k - len(exact)]]\n\n    texts = [\n        f\"{r['name']} {r['category']} {r['description']} {r['opening_hours']} {' '.join(r['events'])}\".lower()\n        for r in records\n    ]\n    model = sentence_model()\n    if model is not None:\n        embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=False)\n        query_vec = model.encode([q], normalize_embeddings=True, show_progress_bar=False)[0]\n        scores = np.asarray(embeddings) @ np.asarray(query_vec)\n    else:\n        query_terms = set(q.split())\n        scores = np.asarray([\n            len(query_terms & set(text.split())) / max(len(query_terms), 1)\n            for text in texts\n        ])\n    order = np.argsort(scores)[::-1][:top_k]\n    return [{\"location\": records[int(i)], \"score\": float(scores[int(i)])} for i in order]\n\n\n@st.cache_resource(show_spinner=\"Loading Whisper...\")\ndef whisper_model():\n    try:\n        import whisper\n\n        return whisper.load_model(\"base\")\n    except Exception:\n        return None\n\n\n@st.cache_resource(show_spinner=\"Loading CLIP...\")\ndef clip_model():\n    try:\n        import torch\n        from transformers import CLIPModel, CLIPProcessor\n\n        device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n        processor = CLIPProcessor.from_pretrained(\"openai/clip-vit-base-patch32\")\n        model = CLIPModel.from_pretrained(\"openai/clip-vit-base-patch32\").to(device)\n        model.eval()\n        return model, processor, device\n    except Exception:\n        return None\n\n\ndef retrieve_image(image: Image.Image, top_k: int = 3) -> list[dict]:\n    records = all_locations()\n    loaded = clip_model()\n    if loaded is None:\n        return [{\"location\": r, \"score\": 0.1} for r in records[:top_k]]\n    import torch\n\n    model, processor, device = loaded\n    prompts = [f\"Photo of {r['name']}, a {r['category']} campus location. {r['description']}\" for r in records]\n    with torch.no_grad():\n        text_inputs = processor(text=prompts, return_tensors=\"pt\", padding=True, truncation=True).to(device)\n        image_inputs = processor(images=image.convert(\"RGB\"), return_tensors=\"pt\").to(device)\n        text_emb = model.get_text_features(**text_inputs)\n        image_emb = model.get_image_features(**image_inputs)\n        text_emb = text_emb / text_emb.norm(dim=-1, keepdim=True)\n        image_emb = image_emb / image_emb.norm(dim=-1, keepdim=True)\n        scores = (text_emb @ image_emb.T).squeeze(1).cpu().numpy()\n    order = np.argsort(scores)[::-1][:top_k]\n    return [{\"location\": records[int(i)], \"score\": float(scores[int(i)])} for i in order]\n\n\ndef save_uploaded(uploaded_file) -> Path:\n    suffix = \".\" + uploaded_file.name.split(\".\")[-1]\n    temp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)\n    temp.write(uploaded_file.getbuffer())\n    temp.flush()\n    return Path(temp.name)\n\n\ndef direction_hint(location: dict) -> str:\n    lat = location[\"coordinates\"][\"lat\"]\n    lon = location[\"coordinates\"][\"lon\"]\n    centre_lat, centre_lon = 51.7540, -1.2540\n    north_south = \"north\" if lat > centre_lat else \"south\"\n    east_west = \"east\" if lon > centre_lon else \"west\"\n    minutes = max(2, int((abs(lat - centre_lat) + abs(lon - centre_lon)) * 9000))\n    return f\"From the central quad, walk {north_south}-{east_west} for about {minutes} minutes.\"\n\n\ndef render_result(result: dict, explanation: str) -> None:\n    location = result[\"location\"]\n    st.markdown('<div class=\"result-panel\">', unsafe_allow_html=True)\n    st.subheader(location[\"name\"])\n    st.caption(location[\"category\"])\n    st.write(location[\"description\"])\n    c1, c2, c3 = st.columns(3)\n    c1.markdown(\"**Opening hours**\")\n    c1.write(location[\"opening_hours\"])\n    c2.markdown(\"**Coordinates**\")\n    c2.write(f\"{location['coordinates']['lat']:.4f}, {location['coordinates']['lon']:.4f}\")\n    c3.markdown(\"**Confidence**\")\n    c3.progress(min(max(result[\"score\"], 0.0), 1.0))\n    c3.write(f\"{result['score']:.2f}\")\n    st.markdown(\"**Directions**\")\n    st.write(direction_hint(location))\n    st.markdown(\"**Events**\")\n    for event in location[\"events\"]:\n        st.write(f\"- {event}\")\n    st.markdown(\"**Retrieval explanation**\")\n    st.info(explanation)\n    st.markdown(\"</div>\", unsafe_allow_html=True)\n    st.map(pd.DataFrame([{\"lat\": location[\"coordinates\"][\"lat\"], \"lon\": location[\"coordinates\"][\"lon\"]}]))\n\n\ninit_db()\n\nif \"chat_history\" not in st.session_state:\n    st.session_state.chat_history = []\n\nwith st.sidebar:\n    st.title(\"Smart Campus\")\n    st.caption(\"Colab multimodal tour assistant\")\n    mode = st.radio(\"Input mode\", [\"Text only\", \"Image only\", \"Voice only\", \"Combined multimodal\"])\n    st.divider()\n    st.markdown(\"**Models**\")\n    st.write(\"CLIP + FAISS-style retrieval\")\n    st.write(\"Whisper speech-to-text\")\n    st.write(\"DistilBERT-compatible intent logic\")\n    st.write(\"Semantic knowledge retrieval\")\n\nst.title(\"Smart Campus Tour & Information Multi-Modal Chatbot\")\nst.write(\"Ask about buildings, directions, opening hours, events, facilities, food, study spaces, and accessibility.\")\n\nleft, right = st.columns([1, 1])\nwith left:\n    image_file = st.file_uploader(\"Upload campus image\", type=[\"png\", \"jpg\", \"jpeg\"]) if mode in {\"Image only\", \"Combined multimodal\"} else None\n    audio_file = st.file_uploader(\"Upload voice query\", type=[\"wav\", \"mp3\", \"m4a\"]) if mode in {\"Voice only\", \"Combined multimodal\"} else None\n    query = st.text_area(\"Enter your question\", placeholder=\"Where is the Library?\") if mode in {\"Text only\", \"Combined multimodal\"} else \"\"\n    run = st.button(\"Ask campus assistant\", type=\"primary\", use_container_width=True)\n\nwith right:\n    df = pd.DataFrame(all_locations())\n    st.subheader(\"Campus locations\")\n    st.dataframe(df[[\"name\", \"category\", \"opening_hours\"]], hide_index=True, use_container_width=True)\n\nif run:\n    transcript = \"\"\n    top3 = []\n    explanation = []\n    if audio_file is not None:\n        model = whisper_model()\n        if model is None:\n            st.error(\"Whisper is unavailable. Run the install cell and ensure ffmpeg is installed.\")\n        else:\n            audio_path = save_uploaded(audio_file)\n            result = model.transcribe(str(audio_path), fp16=False)\n            transcript = result.get(\"text\", \"\").strip()\n            st.success(f\"Transcript: {transcript}\")\n            explanation.append(\"Voice was transcribed using Whisper.\")\n\n    if image_file is not None:\n        image = Image.open(image_file)\n        st.image(image, caption=\"Uploaded image\", use_container_width=True)\n        top3 = retrieve_image(image, top_k=3)\n        explanation.append(\"Image was matched against CLIP text prompts for campus locations.\")\n\n    text_signal = \" \".join([query, transcript]).strip()\n    if text_signal:\n        label, conf = intent(text_signal)\n        st.info(f\"Intent: {label} | Confidence: {conf:.2f}\")\n        semantic_top3 = semantic_search(text_signal, top_k=3)\n        if semantic_top3 and (not top3 or semantic_top3[0][\"score\"] >= top3[0][\"score\"]):\n            top3 = semantic_top3\n        explanation.append(\"Text/transcript was routed through semantic knowledge-base retrieval.\")\n\n    if not top3:\n        st.error(\"No usable input was available.\")\n    else:\n        render_result(top3[0], \" \".join(explanation))\n        st.markdown(\"**Top-3 matches**\")\n        st.dataframe(\n            pd.DataFrame(\n                [{\"Rank\": i + 1, \"Location\": row[\"location\"][\"name\"], \"Score\": round(row[\"score\"], 3)} for i, row in enumerate(top3)]\n            ),\n            hide_index=True,\n            use_container_width=True,\n        )\n        st.session_state.chat_history.append({\"query\": text_signal or \"[image]\", \"location\": top3[0][\"location\"][\"name\"]})\n\nif st.session_state.chat_history:\n    st.divider()\n    st.subheader(\"Chat history\")\n    st.dataframe(pd.DataFrame(st.session_state.chat_history), hide_index=True, use_container_width=True)\n"
        (BASE / 'app_colab.py').write_text(app_code, encoding='utf-8')
        print('Wrote', BASE / 'app_colab.py')
        

## 4. Quick Evaluation Scenarios

In [ ]:
        import pandas as pd
        from pathlib import Path

        BASE = Path('/content/smart_campus_colab')
        RESULTS_DIR = BASE / 'reports' / 'results'
        scenarios = pd.DataFrame([
            {'scenario': 'Library text query', 'input': 'where is Library', 'expected_location': 'Library'},
            {'scenario': 'Innovation event query', 'input': 'what events are happening at eventhub', 'expected_location': 'Innovation Hub'},
            {'scenario': 'Cafeteria food query', 'input': 'where can I get lunch', 'expected_location': 'Cafeteria'},
            {'scenario': 'Accessibility query', 'input': 'I need step free access', 'expected_location': 'Accessibility Office'},
            {'scenario': 'Sports query', 'input': 'where is the gym', 'expected_location': 'Sports Centre'},
        ])
        scenarios.to_csv(RESULTS_DIR / 'testing_results.csv', index=False)
        display(scenarios)
        

## 5. Launch Streamlit in Colab

        Run this cell. It prints a public URL. If LocalTunnel asks for a password, use the IP address printed by the cell.
        Keep the cell running while you use the app.

In [ ]:
        !npm install -g localtunnel > /dev/null 2>&1
        !streamlit run /content/smart_campus_colab/app_colab.py --server.port 8501 > /content/smart_campus_streamlit.log 2>&1 &
        import time, subprocess
        time.sleep(5)
        print('LocalTunnel password/IP, if requested:')
        !curl -s https://ipv4.icanhazip.com
        print('Opening public Streamlit tunnel...')
        !npx localtunnel --port 8501
        

## 6. Optional: Inspect Streamlit Logs

        Run this if the public URL does not open.

In [ ]:
!tail -n 80 /content/smart_campus_streamlit.log